In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle

# ─── 1. LOAD DATA ───────────────────────────────────────────────
df = pd.read_csv("/content/drive/MyDrive/Machine Learning /bigmart.csv")
print("Shape:", df.shape)

# ─── 2. HANDLE MISSING VALUES ───────────────────────────────────
df['Item_Weight'].fillna(df['Item_Weight'].mean(), inplace=True)
df['Outlet_Size'].fillna(df['Outlet_Size'].mode()[0], inplace=True)

# ─── 3. FEATURE ENGINEERING ─────────────────────────────────────
df['Outlet_Age'] = 2013 - df['Outlet_Establishment_Year']

df['Item_Fat_Content'] = df['Item_Fat_Content'].replace({
    'LF': 'Low Fat',
    'low fat': 'Low Fat',
    'reg': 'Regular'
})

# ─── 4. ENCODE CATEGORICAL COLUMNS ──────────────────────────────
le = LabelEncoder()
categorical_cols = ['Item_Identifier', 'Item_Fat_Content', 'Item_Type',
    'Outlet_Identifier', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type'
]

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

#  5. PREPARE FEATURES & TARGET

X = df.drop(['Item_Outlet_Sales', 'Outlet_Establishment_Year'], axis=1)
y = df['Item_Outlet_Sales']
print("Features used:", list(X.columns))

# ─── 6. TRAIN/TEST SPLIT ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ─── 7. TRAIN MODEL ──────────────────────────────────────────────
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# ─── 8. EVALUATE ─────────────────────────────────────────────────
predictions = model.predict(X_test)

mae  = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2   = r2_score(y_test, predictions)
mean_sales = y.mean()
forecast_error = (mae / mean_sales) * 100

print("\n===== KPI RESULTS =====")
print(f"MAE            : {mae:.2f}")
print(f"RMSE           : {rmse:.2f}")
print(f"R² Score       : {r2:.2f}")
print(f"Mean Sales     : {mean_sales:.2f}")
print(f"Forecast Error : {forecast_error:.1f}%")

# ─── 9. FEATURE IMPORTANCE ───────────────────────────────────────
# feat_imp = pd.DataFrame({'Feature': X.columns,'Importance': model.feature_importances_}).sort_values('Importance', ascending=False)
# print("\n===== FEATURE IMPORTANCE =====")
# print(feat_imp.to_string(index=False))

# ─── 10. SAVE MODEL ──────────────────────────────────────────────
pickle.dump(model, open('model.pkl', 'wb'))
print("\nModel saved as model.pkl")

Shape: (8523, 12)
Features used: ['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type', 'Item_MRP', 'Outlet_Identifier', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type', 'Outlet_Age']


/tmp/ipykernel_3134/1062452979.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Item_Weight'].fillna(df['Item_Weight'].mean(), inplace=True)
/tmp/ipykernel_3134/1062452979.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, in


===== KPI RESULTS =====
MAE            : 762.37
RMSE           : 1093.17
R² Score       : 0.56
Mean Sales     : 2181.29
Forecast Error : 35.0%

Model saved as model.pkl
